# BNPL Default Risk – Interactive Inference

## Workflow Overview
This notebook demonstrates how to use the trained PyTorch model to make real‑time predictions on single customer records. The workflow is:

1. **Load the saved model and transformers** – from `models/best_pytorch_model.pth` and `models/transformers/`.
2. **Define feature columns** – numeric and categorical, with acceptable ranges and options.
3. **Interactive input** – the user enters values for each feature, while validation ensures inputs are sensible.
4. **Preprocess** – scaling numerics and encoding categoricals using the saved transformers (fitted on training data).
5. **Predict** – running the model and displaying the predicted risk class (Low/Medium/High) along with class probabilities.
6. **Repeat** – allows the user to make another prediction or exit.

## Why Interactive?
This mimics a real‑world deployment scenario where a business user (e.g., a loan officer) has a new customer's data and wants an instant risk assessment. It also validates that the saved model and preprocessing pipeline work seamlessly outside the training environment.

### Imports and Configuration
Setting up the notebook environment. I import utility functions from `src/utils.py` to keep the notebook concise and maintainable.

In [1]:
import os
os.chdir('..')   # move to project root
print("Working directory:", os.getcwd())

import sys
sys.path.append('./src')
import utils as U
import numpy as np
import torch

Working directory: c:\Users\Alexz\BNPL_ML


### Load Saved Components
Loading the best PyTorch model, the RobustScaler, and the categorical encoders. These were saved during the training pipeline.

In [2]:
model, scaler, cat_mappings = U.load_inference_components()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model loaded on device: {device}")
print("Scaler and category mappings loaded.")

Model loaded on device: cuda
Scaler and category mappings loaded.


### Feature Definitions
I need to know the exact column names and their acceptable ranges/options to guide user input.

In [3]:
num_cols = ['Age', 'Total_BNPL_Active_Loans', 'Total_BNPL_Debt_USD',
            'Average_Transaction_Value_USD', 'Income_USD', 'Credit_Score']
cat_cols = ['Employment_Status', 'Late_Payment_History', 'Shopping_Category_Most_Frequent']

# Acceptable ranges for numerics (based on training data)
ranges = {
    'Age': (18, 100),
    'Total_BNPL_Active_Loans': (0, 10),
    'Total_BNPL_Debt_USD': (0, 50000),
    'Average_Transaction_Value_USD': (0, 2000),
    'Income_USD': (5000, 150000),
    'Credit_Score': (300, 850)
}

# Options for categoricals (from training categories)
cat_options = {}
for col in cat_cols:
    options = list(cat_mappings[col].keys())
    cat_options[col] = optionsnum_cols = ['Age', 'Total_BNPL_Active_Loans', 'Total_BNPL_Debt_USD',
            'Average_Transaction_Value_USD', 'Income_USD', 'Credit_Score']
cat_cols = ['Employment_Status', 'Late_Payment_History', 'Shopping_Category_Most_Frequent']

# Acceptable ranges for numerics (based on training data)
ranges = {
    'Age': (18, 100),
    'Total_BNPL_Active_Loans': (0, 10),
    'Total_BNPL_Debt_USD': (0, 50000),
    'Average_Transaction_Value_USD': (0, 2000),
    'Income_USD': (5000, 150000),
    'Credit_Score': (300, 850)
}

# Options for categoricals (from training categories)
cat_options = {}
for col in cat_cols:
    options = list(cat_mappings[col].keys())
    cat_options[col] = options

### Gather User Input
The notebook prompts the user to enter values for each feature. For numeric fields, it validates that the input is within the acceptable range. For categoricals, it lists the possible choices. The user can type the exact option or leave it blank, which will default to the first category.

In [4]:
def get_user_input():
    input_dict = {}
    print("\n" + "="*60)
    print("Enter customer details (type the value and press Enter)")
    print("For numeric fields, values must be within the shown range.")
    print("For categorical fields, choose from the listed options.")
    print("="*60)
    
    # Numeric fields
    for col in num_cols:
        low, high = ranges[col]
        while True:
            try:
                val = input(f"{col} [{low}-{high}]: ")
                if val.strip() == '':
                    raise ValueError("Value cannot be empty.")
                num_val = float(val)
                if num_val < low or num_val > high:
                    print(f"  Value out of range. Please enter between {low} and {high}.")
                    continue
                input_dict[col] = num_val
                break
            except ValueError as e:
                print(f"  Invalid input: {e}. Please enter a number.")
    
    # Categorical fields
    for col in cat_cols:
        options = cat_options[col]
        print(f"\n{col} – options: {options}")
        while True:
            val = input(f"{col}: ")
            if val.strip() == '':
                print("  Value cannot be empty.")
                continue
            if val in options:
                input_dict[col] = val
                break
            else:
                print(f"  '{val}' not in options. Please choose from: {options}")
    return input_dict

### Preprocess and Predict
Once the user has entered all values, I preprocess the data (scale numerics, encode categoricals) and run the model. The predicted class and the probability for each class are displayed.

In [ ]:
# Get input
user_data = get_user_input()

# Add engineered feature
user_data['Debt_to_Income_Ratio'] = (
    user_data['Total_BNPL_Debt_USD'] / user_data['Income_USD']
)

# Use num_cols with the derived feature to match the model's expected input
num_cols_with_ratio = num_cols + ['Debt_to_Income_Ratio']

# Warn if the scaler was fit on a different column order
if hasattr(scaler, "feature_names_in_"):
    assert list(scaler.feature_names_in_) == num_cols_with_ratio, (
        f"Column mismatch:\n  scaler fit on: {list(scaler.feature_names_in_)}\n"
        f"  passing      : {num_cols_with_ratio}"
    )

# Preprocess
X_num, X_cat = U.preprocess_single_sample(
    user_data, num_cols_with_ratio, cat_cols, scaler, cat_mappings
)

# Predict
pred_class, probs = U.predict_single(model, X_num, X_cat, device)

# Map class to label
target_names = ['Low', 'Medium', 'High']
pred_label = target_names[pred_class]

print("\n" + "="*60)
print("INPUT PARAMETERS CONFIRMED")
print("="*60)
for key in num_cols_with_ratio + cat_cols:
    v = user_data[key]
    if isinstance(v, float):
        v = f"{v:.4f}" if key == 'Debt_to_Income_Ratio' else f"{v:g}"
    print(f"  {key:<28}: {v}")

# Display results
print("\n" + "="*60)
print("PREDICTION RESULT")
print("="*60)
print(f"Predicted Default Risk: {pred_label}")
print("\nClass Probabilities:")
for label, prob in zip(target_names, probs):
    print(f"  {label}: {prob*100:.2f}%")
print("="*60)


Enter customer details (type the value and press Enter)
For numeric fields, values must be within the shown range.
For categorical fields, choose from the listed options.

Employment_Status – options: ['Employed', 'Freelancer', 'Student', 'Unemployed']

Late_Payment_History – options: ['No', 'Yes']

Shopping_Category_Most_Frequent – options: ['Electronics', 'Fashion', 'Groceries/Essentials', 'Home/Furniture', 'Travel']

INPUT PARAMETERS CONFIRMED
  Age                         : 56
  Total_BNPL_Active_Loans     : 7
  Total_BNPL_Debt_USD         : 6592
  Average_Transaction_Value_USD: 329
  Income_USD                  : 129811
  Credit_Score                : 600
  Debt_to_Income_Ratio        : 0.0508
  Employment_Status           : Employed
  Late_Payment_History        : No
  Shopping_Category_Most_Frequent: Electronics

PREDICTION RESULT
Predicted Default Risk: Medium

Class Probabilities:
  Low: 0.69%
  Medium: 53.49%
  High: 45.82%


### Another Prediction?
The user can choose to run another prediction or exit the notebook.

In [6]:
while True:
    again = input("\nMake another prediction? (y/n): ").strip().lower()
    if again in ['y', 'yes']:
        user_data = get_user_input()
        X_num, X_cat = U.preprocess_single_sample(user_data, num_cols, cat_cols, scaler, cat_mappings)
        pred_class, probs = U.predict_single(model, X_num, X_cat, device)
        pred_label = target_names[pred_class]
        print("\n" + "="*60)
        print(f"Predicted Default Risk: {pred_label}")
        print("\nClass Probabilities:")
        for label, prob in zip(target_names, probs):
            print(f"  {label}: {prob*100:.2f}%")
        print("="*60)
    else:
        print("Exiting inference.")
        break

Exiting inference.


## Conclusion
This notebook demonstrates a complete inference pipeline using a saved PyTorch model. The interactive interface allows quick and simple ad‑hoc predictions, keeping it user-friendly for later UI integration. It also validates that the preprocessing steps (scaling and categorical encoding) are applied consistently with the training pipeline, which is a critical requirement for a deployable model.